In [ ]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface langchain-community pypdf sentence-transformers faiss-cpu langgraph reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

print("✅ Setup complete")

/tmp/ipykernel_4803/2136603820.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Setup complete


In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

def create_pdf(filename, title, content):
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, height-50, title)
    c.setFont("Helvetica", 9)
    y = height - 80
    for line in content:
        if y < 50:
            c.showPage()
            y = height - 50
        c.drawString(50, y, line)
        y -= 14
    c.save()
    print(f"✅ Created: {filename}")

create_pdf("esg_policy.pdf", "NORTHERN EUROPE BANK — ESG POLICY 2024", [
    "Section 1 — Scope",
    "Applies to corporate clients with turnover exceeding EUR 10 million.",
    "Section 2 — Reporting Requirements",
    "- GHG Scope 1 and 2 emissions in tonnes CO2",
    "- Energy consumption in MWh",
    "- Water usage in cubic metres",
    "Section 3 — EU Taxonomy",
    "DNSH assessment mandatory for Energy, Manufacturing and Transport.",
    "Section 4 — Deadlines",
    "Annual ESG reports due 31 March each year.",
    "Section 5 — Non-Compliance",
    "- Credit facility review",
    "- Potential loan covenant breach",
    "- Escalation to senior management",
])

create_pdf("credit_risk_policy.pdf", "NORTHERN EUROPE BANK — CREDIT RISK POLICY 2024", [
    "Section 1 — Risk Classification",
    "Green: score above 750 — standard terms",
    "Amber: score 650-750 — enhanced monitoring",
    "Red: score below 650 — credit committee approval",
    "Section 2 — Exposure Limits",
    "Single client exposure must not exceed 10% of total portfolio.",
    "Sector concentration limit is 25% of total portfolio.",
    "Real estate sector limit is 30%.",
    "Section 3 — Review Frequency",
    "Green clients: Annual review",
    "Amber clients: Semi-annual review",
    "Red clients: Quarterly review",
    "Section 4 — Reporting",
    "Credit risk reports due to Risk Committee monthly.",
    "Exceptions above EUR 5 million require immediate escalation.",
])

print("\n✅ Policy documents ready")

✅ Created: esg_policy.pdf
✅ Created: credit_risk_policy.pdf

✅ Policy documents ready


In [4]:
# Load both documents
pdf_files = ["esg_policy.pdf", "credit_risk_policy.pdf"]
all_documents = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = pdf
    all_documents.extend(docs)
    print(f"✅ Loaded: {pdf}")

# Create vector store
all_chunks = splitter.split_documents(all_documents)
vectorstore = FAISS.from_documents(all_chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"\n✅ Vector store ready — {len(all_chunks)} chunks")

✅ Loaded: esg_policy.pdf
✅ Loaded: credit_risk_policy.pdf

✅ Vector store ready — 8 chunks


In [5]:
# This is what makes it CORRECTIVE RAG
# We grade whether retrieved chunks are actually relevant

grader_prompt = ChatPromptTemplate.from_template("""
You are a relevance grader.
Assess whether the retrieved document is relevant to the question.

Retrieved document:
{document}

Question: {question}

Give a binary score:
- "yes" if the document is relevant to the question
- "no" if the document is NOT relevant

Return ONLY "yes" or "no" — nothing else.""")

grader_chain = grader_prompt | llm | StrOutputParser()

def grade_document(document, question):
    """Grade if a document chunk is relevant to the question"""
    score = grader_chain.invoke({
        "document": document.page_content,
        "question": question
    })
    return score.strip().lower() == "yes"

# Test the grader
from langchain_core.documents import Document

test_doc = Document(page_content="Annual ESG reports must be submitted by 31 March each year.")
test_question = "What is the ESG reporting deadline?"

result = grade_document(test_doc, test_question)
print(f"✅ Grader working!")
print(f"   Document: 'ESG reports due 31 March'")
print(f"   Question: 'What is the ESG reporting deadline?'")
print(f"   Relevant: {result}")

# Test with irrelevant document
test_doc2 = Document(page_content="Red clients require quarterly review and enhanced due diligence.")
result2 = grade_document(test_doc2, test_question)
print(f"\n   Document: 'Red clients quarterly review'")
print(f"   Question: 'What is the ESG reporting deadline?'")
print(f"   Relevant: {result2}")

✅ Grader working!
   Document: 'ESG reports due 31 March'
   Question: 'What is the ESG reporting deadline?'
   Relevant: True

   Document: 'Red clients quarterly review'
   Question: 'What is the ESG reporting deadline?'
   Relevant: False


In [6]:
# State for our Corrective RAG graph
class CorrectiveRAGState(TypedDict):
    question: str
    documents: List
    filtered_documents: List
    answer: str
    search_attempts: int

# Node 1 — Retrieve documents
def retrieve(state: CorrectiveRAGState):
    print(f"🔍 Retrieving documents...")
    question = state["question"]
    documents = retriever.invoke(question)
    print(f"   Retrieved {len(documents)} chunks")
    return {"documents": documents, "search_attempts": state.get("search_attempts", 0) + 1}

# Node 2 — Grade documents
def grade_documents(state: CorrectiveRAGState):
    print(f"📊 Grading relevance...")
    question = state["question"]
    documents = state["documents"]

    filtered = []
    for doc in documents:
        is_relevant = grade_document(doc, question)
        source = doc.metadata.get("source", "unknown")
        if is_relevant:
            filtered.append(doc)
            print(f"   ✅ Relevant: {source[:30]}...")
        else:
            print(f"   ❌ Not relevant: {source[:30]}...")

    print(f"   Kept {len(filtered)}/{len(documents)} chunks")
    return {"filtered_documents": filtered}

# Node 3 — Generate answer
def generate(state: CorrectiveRAGState):
    print(f"🤖 Generating answer...")
    question = state["question"]
    documents = state["filtered_documents"]

    context = "\n\n".join([
        f"[{doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in documents
    ])

    answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the provided context.
Always cite which document your answer comes from.
If context is insufficient say so clearly.

Context:
{context}

Question: {question}

Answer:""")

    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({"context": context, "question": question})

    return {"answer": answer}

# Node 4 — Rewrite question (when no relevant docs found)
def rewrite_question(state: CorrectiveRAGState):
    print(f"✏️  Rewriting question for better search...")
    question = state["question"]

    rewrite_prompt = ChatPromptTemplate.from_template("""
The original question did not retrieve relevant documents.
Rewrite it to be more specific and likely to find relevant information.

Original question: {question}

Rewritten question (return ONLY the rewritten question):""")

    rewrite_chain = rewrite_prompt | llm | StrOutputParser()
    new_question = rewrite_chain.invoke({"question": question})

    print(f"   Original: {question}")
    print(f"   Rewritten: {new_question}")

    return {"question": new_question}

print("✅ All nodes defined")

✅ All nodes defined


In [7]:
def decide_next_step(state: CorrectiveRAGState):
    """
    After grading — what do we do next?

    If we have relevant docs → generate answer
    If no relevant docs AND first attempt → rewrite and retry
    If no relevant docs AND already retried → generate anyway
    """
    filtered = state["filtered_documents"]
    attempts = state.get("search_attempts", 1)

    if len(filtered) > 0:
        print(f"   → Has {len(filtered)} relevant docs — generating answer")
        return "generate"
    elif attempts < 2:
        print(f"   → No relevant docs — rewriting question")
        return "rewrite"
    else:
        print(f"   → No relevant docs after retry — generating with what we have")
        state["filtered_documents"] = state["documents"]
        return "generate"

print("✅ Routing logic defined")

✅ Routing logic defined


In [8]:
# Build Corrective RAG graph
graph = StateGraph(CorrectiveRAGState)

# Add all nodes
graph.add_node("retrieve", retrieve)
graph.add_node("grade_documents", grade_documents)
graph.add_node("generate", generate)
graph.add_node("rewrite_question", rewrite_question)

# Set entry point
graph.set_entry_point("retrieve")

# Edges
graph.add_edge("retrieve", "grade_documents")

# After grading — conditional routing
graph.add_conditional_edges(
    "grade_documents",
    decide_next_step,
    {
        "generate": "generate",
        "rewrite": "rewrite_question"
    }
)

# After rewrite → retrieve again
graph.add_edge("rewrite_question", "retrieve")

# After generate → END
graph.add_edge("generate", END)

# Compile
corrective_rag = graph.compile()

print("✅ Corrective RAG graph compiled!")
print("""
Graph flow:

START
  ↓
[retrieve] → [grade_documents]
                ↓
        relevant docs?
        YES → [generate] → END
        NO  → [rewrite_question] → [retrieve] → [grade_documents]
                                    (retry once, then generate anyway)
""")

✅ Corrective RAG graph compiled!

Graph flow:

START
  ↓
[retrieve] → [grade_documents]
                ↓
        relevant docs?
        YES → [generate] → END
        NO  → [rewrite_question] → [retrieve] → [grade_documents]
                                    (retry once, then generate anyway)



In [9]:
def run_corrective_rag(question):
    print(f"\n{'='*55}")
    print(f"❓ Question: {question}")
    print(f"{'='*55}")

    initial_state = {
        "question": question,
        "documents": [],
        "filtered_documents": [],
        "answer": "",
        "search_attempts": 0
    }

    final_state = corrective_rag.invoke(initial_state)

    print(f"\n💬 ANSWER:")
    print(final_state["answer"])
    print(f"\n📊 Search attempts: {final_state['search_attempts']}")
    return final_state

print("✅ Run function ready")

✅ Run function ready


In [10]:
# Test 1 — Should find relevant docs immediately
run_corrective_rag("What is the deadline for ESG reporting?")


❓ Question: What is the deadline for ESG reporting?
🔍 Retrieving documents...
   Retrieved 3 chunks
📊 Grading relevance...
   ✅ Relevant: esg_policy.pdf...
   ❌ Not relevant: esg_policy.pdf...
   ❌ Not relevant: credit_risk_policy.pdf...
   Kept 1/3 chunks
   → Has 1 relevant docs — generating answer
🤖 Generating answer...

💬 ANSWER:
The deadline for ESG reporting is 31 March each year. [esg_policy.pdf, Section 4 — Deadlines]

📊 Search attempts: 1


{'question': 'What is the deadline for ESG reporting?',
 'documents': [Document(id='1ee177f6-ed00-4179-a0d4-e70c6986b608', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:01+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:01+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'esg_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Section 3 — EU Taxonomy\nDNSH assessment mandatory for Energy, Manufacturing and Transport.\nSection 4 — Deadlines\nAnnual ESG reports due 31 March each year.\nSection 5 — Non-Compliance'),
  Document(id='bf76b1b8-5163-4cc7-b945-68602be2d851', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:01+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:01+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trappe

In [11]:
# Test 2 — Vague question that might trigger rewrite
run_corrective_rag("When do things need to be submitted?")


❓ Question: When do things need to be submitted?
🔍 Retrieving documents...
   Retrieved 3 chunks
📊 Grading relevance...
   ✅ Relevant: esg_policy.pdf...
   ✅ Relevant: credit_risk_policy.pdf...
   ❌ Not relevant: credit_risk_policy.pdf...
   Kept 2/3 chunks
   → Has 2 relevant docs — generating answer
🤖 Generating answer...

💬 ANSWER:
According to the provided context, the following submissions are required:

1. Annual ESG reports: due 31 March each year ([esg_policy.pdf], Section 4 — Deadlines)
2. Credit risk reports: due to Risk Committee monthly ([credit_risk_policy.pdf], Section 4 — Reporting)
3. Exceptions above EUR 5 million: require immediate escalation ([credit_risk_policy.pdf], Section 4 — Reporting)
4. DNSH assessment: no specific deadline mentioned, but it is mandatory for Energy, Manufacturing and Transport ([esg_policy.pdf], Section 3 — EU Taxonomy)
5. Client reviews: 
    - Amber clients: semi-annual review ([credit_risk_policy.pdf])
    - Red clients: quarterly review (

{'question': 'When do things need to be submitted?',
 'documents': [Document(id='1ee177f6-ed00-4179-a0d4-e70c6986b608', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:01+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:01+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'esg_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Section 3 — EU Taxonomy\nDNSH assessment mandatory for Energy, Manufacturing and Transport.\nSection 4 — Deadlines\nAnnual ESG reports due 31 March each year.\nSection 5 — Non-Compliance'),
  Document(id='4e0941c6-caf6-4843-a095-5dceafd38344', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:02+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:02+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped':

In [12]:
# Test 3 — Needs both documents
run_corrective_rag("What are all the deadlines and review frequencies in our policies?")


❓ Question: What are all the deadlines and review frequencies in our policies?
🔍 Retrieving documents...
   Retrieved 3 chunks
📊 Grading relevance...
   ✅ Relevant: esg_policy.pdf...
   ✅ Relevant: credit_risk_policy.pdf...
   ✅ Relevant: credit_risk_policy.pdf...
   Kept 3/3 chunks
   → Has 3 relevant docs — generating answer
🤖 Generating answer...

💬 ANSWER:
According to the provided context, the deadlines and review frequencies are as follows:

1. Annual ESG reports are due on 31 March each year ([esg_policy.pdf], Section 4 — Deadlines).
2. Green clients are reviewed annually ([credit_risk_policy.pdf], Section 3 — Review Frequency).
3. Amber clients are reviewed semi-annually ([credit_risk_policy.pdf], two instances, but same information).
4. Red clients are reviewed quarterly ([credit_risk_policy.pdf], second instance).
5. Credit risk reports are due to the Risk Committee monthly ([credit_risk_policy.pdf], Section 4 — Reporting).
6. Exceptions above EUR 5 million require immediate

{'question': 'What are all the deadlines and review frequencies in our policies?',
 'documents': [Document(id='1ee177f6-ed00-4179-a0d4-e70c6986b608', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:01+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:01+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'esg_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Section 3 — EU Taxonomy\nDNSH assessment mandatory for Energy, Manufacturing and Transport.\nSection 4 — Deadlines\nAnnual ESG reports due 31 March each year.\nSection 5 — Non-Compliance'),
  Document(id='aecb0284-320a-486d-9308-7304b1696bbb', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-07-31T23:14:02+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-07-31T23:14:02+00:00', 'subject': 'unspecified', '

In [13]:
# Show the difference clearly
print("📊 REGULAR RAG vs CORRECTIVE RAG")
print("=" * 55)

test_question = "What happens to high risk clients?"

# Regular RAG — no grading
print("\n🔴 REGULAR RAG:")
regular_docs = retriever.invoke(test_question)
print(f"Retrieved {len(regular_docs)} chunks — used ALL regardless of relevance")
for doc in regular_docs:
    print(f"  → {doc.page_content[:80]}...")

# Corrective RAG — with grading
print("\n🟢 CORRECTIVE RAG:")
state = run_corrective_rag(test_question)

📊 REGULAR RAG vs CORRECTIVE RAG

🔴 REGULAR RAG:
Retrieved 3 chunks — used ALL regardless of relevance
  → Amber clients: Semi-annual review
Red clients: Quarterly review
Section 4 — Repo...
  → NORTHERN EUROPE BANK — ESG POLICY 2024
Section 1 — Scope
Applies to corporate cl...
  → Sector concentration limit is 25% of total portfolio.
Real estate sector limit i...

🟢 CORRECTIVE RAG:

❓ Question: What happens to high risk clients?
🔍 Retrieving documents...
   Retrieved 3 chunks
📊 Grading relevance...
   ✅ Relevant: credit_risk_policy.pdf...
   ❌ Not relevant: esg_policy.pdf...
   ❌ Not relevant: credit_risk_policy.pdf...
   Kept 1/3 chunks
   → Has 1 relevant docs — generating answer
🤖 Generating answer...

💬 ANSWER:
According to the [credit_risk_policy.pdf] document, high risk clients, referred to as "Red clients", undergo a Quarterly review. (Source: [credit_risk_policy.pdf])

📊 Search attempts: 1


In [14]:
print("📄 CORRECTIVE RAG — Policy Assistant")
print("Searches ESG and Credit Risk policies")
print("Automatically retries if wrong chunks found")
print("Type 'exit' to quit\n")

while True:
    question = input("❓ Your question: ")
    if question.lower() == 'exit':
        print("👋 Goodbye!")
        break
    run_corrective_rag(question)

📄 CORRECTIVE RAG — Policy Assistant
Searches ESG and Credit Risk policies
Automatically retries if wrong chunks found
Type 'exit' to quit

❓ Your question: exit
👋 Goodbye!
